# 🚀 HTR Training: CRNN + CTC Loss (Local Version)

## Training Pipeline untuk Handwritten Text Recognition

### Arsitektur Model:
```
Input Image (64 x W x 1)
    ↓
CNN Backbone (VGG-like, 512 channels)
    ↓
Bidirectional LSTM (2 layers, 256 hidden)
    ↓
Linear → CTC Loss
```

### 🔄 Training Flow (2-Phase):

```
┌─────────────────────────────────────────────────────────────┐
│  PHASE 1: PRE-TRAINING (Synthetic Data)                     │
│  Dataset: 1,000,000 synthetic line images                   │
│  Epochs: 5                                                  │
│  LR: 1e-3                                                   │
│  Goal: Learn general handwriting features                   │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  PHASE 2: FINE-TUNING (IAM Real Data)                       │
│  Dataset: ~6,161 IAM train lines (Aachen split)             │
│  Epochs: 100                                                │
│  LR: 1e-4                                                   │
│  Goal: Adapt to real handwriting                            │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  EVALUATION (IAM Test)                                      │
│  Dataset: ~1,861 IAM test lines                             │
│  Metrics: CER, WER                                          │
└─────────────────────────────────────────────────────────────┘
```

### 📊 Dataset Summary:

| Phase | Dataset | Split | Samples |
|-------|---------|-------|---------|
| Pre-train | Synthetic | Train | 900,000 |
| Pre-train | Synthetic | Val | 50,000 |
| Fine-tune | IAM | Train | ~6,161 |
| Fine-tune | IAM | Val | ~966 |
| Evaluate | IAM | Test | ~1,861 |

### 📋 Local Training Settings:
- **GPU**: NVIDIA GPU dengan CUDA
- **Python**: 3.8+
- **PyTorch**: 2.0+

---

## Section 1: Setup & Installation (Local)

In [2]:
# ============================================
# INSTALL DEPENDENCIES (Local)
# ============================================

# Uncomment jika belum install:
# !pip install rapidfuzz albumentations datasets
# Note: rapidfuzz digunakan sebagai pengganti editdistance (support Python 3.13)

import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")

✅ PyTorch version: 2.7.1+cu118
✅ CUDA available: True
   GPU: NVIDIA RTX A2000 12GB
   Memory: 12.9 GB


In [3]:
# ============================================
# IMPORTS
# ============================================

import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, ReduceLROnPlateau

import cv2
from PIL import Image
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Gunakan rapidfuzz (support Python 3.13) sebagai pengganti editdistance
from rapidfuzz.distance import Levenshtein

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

d:\d121221080_HTR_ta\Tugas_Akhir\htr_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch version: 2.7.1+cu118
✅ Device: cuda
   GPU: NVIDIA RTX A2000 12GB
   Memory: 12.9 GB


d:\d121221080_HTR_ta\Tugas_Akhir\htr_env\Lib\site-packages\albumentations\check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)>
  data = fetch_version_info()


In [4]:
# ============================================
# ENVIRONMENT SETUP (LOCAL)
# ============================================

import os
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("🖥️  ENVIRONMENT: LOCAL")
print("=" * 60)

# ===== CONFIGURE YOUR PATHS HERE =====
# ⚠️ UPDATE PATHS SESUAI LOKASI DATA ANDA!

# Base directory (folder tempat notebook ini berada)
BASE_DIR = Path(r"D:\d121221080_HTR_ta\Tugas_Akhir")

# Synthetic Data Directory (hasil dari Synthetic_Line_HTR_Generator.ipynb)
SYNTHETIC_DATA_DIR = BASE_DIR / "synthetic_dataset" / "synthetic_lines_1M" / "synthetic_lines_1M"

# IAM Data Directory (opsional - bisa juga download dari HuggingFace)
# Set None jika ingin download dari HuggingFace
IAM_DATA_DIR = None  # atau Path(r"D:\path\to\iam\data")

# ===== OUTPUT DIRECTORIES =====
EXPERIMENT_NAME = f"HTR_CRNN_Local_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = BASE_DIR / "training_output"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULTS_DIR = OUTPUT_DIR / "results"

for d in [OUTPUT_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===== CHECK SYNTHETIC DATASET =====
print(f"\n🔍 Checking required datasets:")

if SYNTHETIC_DATA_DIR.exists():
    print(f"   ✅ Synthetic data: {SYNTHETIC_DATA_DIR}")
    if (SYNTHETIC_DATA_DIR / "train_synthetic.csv").exists():
        print(f"      └── train_synthetic.csv found")
    else:
        print(f"      ❌ train_synthetic.csv NOT FOUND")
else:
    print(f"   ❌ Synthetic data NOT FOUND: {SYNTHETIC_DATA_DIR}")
    print(f"      ⚠️ Run Synthetic_Line_HTR_Generator.ipynb first!")

if IAM_DATA_DIR:
    print(f"   ℹ️  IAM Data: {IAM_DATA_DIR}")
else:
    print(f"   ℹ️  IAM Lines: Will be downloaded from Hugging Face")

print(f"\n📁 Output: {OUTPUT_DIR}")
print(f"📁 Checkpoints: {CHECKPOINT_DIR}")
print(f"📁 Results: {RESULTS_DIR}")

🖥️  ENVIRONMENT: LOCAL

🔍 Checking required datasets:
   ✅ Synthetic data: D:\d121221080_HTR_ta\Tugas_Akhir\synthetic_dataset\synthetic_lines_1M\synthetic_lines_1M
      └── train_synthetic.csv found
   ℹ️  IAM Lines: Will be downloaded from Hugging Face

📁 Output: D:\d121221080_HTR_ta\Tugas_Akhir\training_output
📁 Checkpoints: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\checkpoints
📁 Results: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\results


In [5]:
# ============================================
# REPRODUCIBILITY (Fixed for Windows)
# ============================================

import torch.backends.cudnn as cudnn

def set_seed(seed: int = 42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # PENTING untuk Windows: jangan set deterministic=True, akan sangat lambat!
    cudnn.deterministic = False  # Trade-off untuk speed
    cudnn.benchmark = True       # Enable untuk consistent input sizes

set_seed(42)
print("✅ Random seeds set")
print("   cudnn.benchmark = True (untuk kecepatan)")
print("   cudnn.deterministic = False (trade-off speed vs reproducibility)")

✅ Random seeds set
   cudnn.benchmark = True (untuk kecepatan)
   cudnn.deterministic = False (trade-off speed vs reproducibility)


## Section 2: Configuration

In [6]:
from dataclasses import dataclass, field, asdict
import json

@dataclass
class TrainingConfig:
    """
    Configuration for HTR 2-Phase Training (Local Version)
    
    Phase 1: Pre-training on Synthetic (high LR, few epochs)
    Phase 2: Fine-tuning on IAM (low LR, many epochs)
    """
    
    # ===== Model Architecture =====
    image_height: int = 64          # Tinggi gambar input
    image_width: int = 512          # Lebar maksimum (line-level)
    
    # CNN backbone
    cnn_output_channels: int = 512
    
    # RNN (LSTM)
    rnn_hidden_size: int = 256
    rnn_num_layers: int = 2
    rnn_dropout: float = 0.3
    bidirectional: bool = True
    
    # ===== PHASE 1: Pre-training on Synthetic =====
    pretrain_epochs: int = 3        # Epochs untuk pre-training
    pretrain_lr: float = 1e-3       # Learning rate tinggi
    pretrain_batch_size: int = 32   # Batch size (sesuaikan dengan GPU memory)
    
    # ===== PHASE 2: Fine-tuning on IAM =====
    finetune_epochs: int = 100      # Epochs untuk fine-tuning
    finetune_lr: float = 1e-4       # Learning rate rendah
    finetune_batch_size: int = 16   # Batch size lebih kecil
    
    # ===== Optimizer Settings =====
    weight_decay: float = 1e-4
    grad_clip: float = 5.0
    
    # Scheduler
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    min_lr: float = 1e-6
    
    # Early stopping
    early_stop_patience: int = 15
    
    # ===== Data Settings =====
    # ⚠️ PENTING: num_workers=0 untuk Windows! Mencegah hang/deadlock
    num_workers: int = 0            # SET 0 UNTUK WINDOWS!
    pin_memory: bool = True
    augment_train: bool = True
    
    # ===== Experiment =====
    experiment_name: str = ""
    random_seed: int = 42
    
    def save_to_json(self, path: str):
        with open(path, 'w') as f:
            json.dump(asdict(self), f, indent=2)
    
    @classmethod
    def load_from_json(cls, path: str) -> 'TrainingConfig':
        with open(path, 'r') as f:
            data = json.load(f)
        return cls(**data)

# Create configuration
config = TrainingConfig(experiment_name=EXPERIMENT_NAME)

# Adjust batch size based on available GPU memory
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_memory < 6:  # Less than 6GB
        config.pretrain_batch_size = 16
        config.finetune_batch_size = 8
        print(f"⚠️ Low GPU memory ({gpu_memory:.1f}GB), reducing batch sizes")
    elif gpu_memory < 12:  # 6-12GB
        config.pretrain_batch_size = 32
        config.finetune_batch_size = 16
    else:  # 12GB+
        config.pretrain_batch_size = 64
        config.finetune_batch_size = 32

# Save config
config.save_to_json(str(RESULTS_DIR / "experiment_config.json"))

print("=" * 60)
print("📋 TRAINING CONFIGURATION")
print("=" * 60)

print(f"\n🏗️  Model Architecture:")
print(f"   Image size: {config.image_height} x {config.image_width}")
print(f"   CNN output: {config.cnn_output_channels} channels")
print(f"   RNN: {config.rnn_num_layers} x Bi-LSTM({config.rnn_hidden_size})")

print(f"\n📈 PHASE 1 - Pre-training (Synthetic):")
print(f"   Epochs: {config.pretrain_epochs}")
print(f"   Batch size: {config.pretrain_batch_size}")
print(f"   Learning rate: {config.pretrain_lr}")

print(f"\n📈 PHASE 2 - Fine-tuning (IAM):")
print(f"   Epochs: {config.finetune_epochs}")
print(f"   Batch size: {config.finetune_batch_size}")
print(f"   Learning rate: {config.finetune_lr}")

print(f"\n⚠️  PENTING (Windows Fix):")
print(f"   num_workers: {config.num_workers} (mencegah hang)")
print(f"\n⏹️  Early Stopping: {config.early_stop_patience} epochs patience")

📋 TRAINING CONFIGURATION

🏗️  Model Architecture:
   Image size: 64 x 512
   CNN output: 512 channels
   RNN: 2 x Bi-LSTM(256)

📈 PHASE 1 - Pre-training (Synthetic):
   Epochs: 3
   Batch size: 64
   Learning rate: 0.001

📈 PHASE 2 - Fine-tuning (IAM):
   Epochs: 100
   Batch size: 32
   Learning rate: 0.0001

⚠️  PENTING (Windows Fix):
   num_workers: 0 (mencegah hang)

⏹️  Early Stopping: 15 epochs patience


## Section 3: Character Encoding

In [7]:
class CharacterEncoder:
    """
    Handles character to index encoding/decoding for CTC
    
    CTC Loss membutuhkan index 0 sebagai BLANK token.
    """
    
    def __init__(self, charset: Optional[Dict[str, int]] = None):
        if charset:
            self.char_to_idx = charset
        else:
            # Default charset - akan di-override dari charset.json
            import string
            chars = list(string.ascii_letters + string.digits + " .,;:'\"!?-()&")
            self.char_to_idx = {'<BLANK>': 0}
            for i, c in enumerate(chars):
                self.char_to_idx[c] = i + 1
        
        self.idx_to_char = {v: k for k, v in self.char_to_idx.items()}
        self.blank_idx = 0
        self.num_classes = len(self.char_to_idx)
    
    def encode(self, text: str) -> List[int]:
        """Convert text to list of indices"""
        encoded = []
        for c in text:
            if c in self.char_to_idx:
                encoded.append(self.char_to_idx[c])
            # Skip unknown characters (already filtered during generation)
        return encoded
    
    def decode(self, indices: List[int], remove_blanks: bool = True) -> str:
        """Convert indices back to text with CTC decoding"""
        chars = []
        prev_idx = None
        
        for idx in indices:
            if remove_blanks:
                # CTC decoding: remove blanks and repeated characters
                if idx != self.blank_idx and idx != prev_idx:
                    char = self.idx_to_char.get(idx, '')
                    if char != '<BLANK>':
                        chars.append(char)
            else:
                chars.append(self.idx_to_char.get(idx, ''))
            prev_idx = idx
        
        return ''.join(chars)
    
    def decode_batch(self, batch_indices: torch.Tensor) -> List[str]:
        """Decode a batch of predictions"""
        results = []
        for indices in batch_indices:
            results.append(self.decode(indices.tolist()))
        return results
    
    @classmethod
    def from_json(cls, path: str) -> 'CharacterEncoder':
        """Load charset from JSON file"""
        with open(path, 'r', encoding='utf-8') as f:
            charset = json.load(f)
        return cls(charset)
    
    def save_json(self, path: str):
        """Save charset to JSON file"""
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(self.char_to_idx, f, indent=2, ensure_ascii=False)
    
    def get_charset_info(self) -> Dict:
        """Get charset statistics for documentation"""
        chars = [c for c in self.char_to_idx.keys() if c != '<BLANK>']
        return {
            'total_classes': self.num_classes,
            'blank_idx': self.blank_idx,
            'num_letters': sum(1 for c in chars if c.isalpha()),
            'num_digits': sum(1 for c in chars if c.isdigit()),
            'num_punctuation': sum(1 for c in chars if not c.isalnum() and c != ' '),
            'has_space': ' ' in self.char_to_idx,
            'sample_chars': chars[:20]
        }

# Load encoder dari charset.json yang sudah ada
CHARSET_PATH = SYNTHETIC_DATA_DIR / "charset.json"

if CHARSET_PATH.exists():
    encoder = CharacterEncoder.from_json(str(CHARSET_PATH))
    print(f"✅ Loaded charset from: {CHARSET_PATH}")
else:
    print("⚠️ charset.json not found, creating default charset...")
    encoder = CharacterEncoder()

# Simpan copy charset ke output dir
encoder.save_json(str(RESULTS_DIR / "charset.json"))

charset_info = encoder.get_charset_info()
print("=" * 60)
print("📝 CHARACTER ENCODER")
print("=" * 60)
print(f"   Total classes: {charset_info['total_classes']} (including BLANK)")
print(f"   Letters: {charset_info['num_letters']}")
print(f"   Digits: {charset_info['num_digits']}")
print(f"   Punctuation: {charset_info['num_punctuation']}")
print(f"   Has space: {charset_info['has_space']}")
print(f"\n   Sample chars: {''.join(charset_info['sample_chars'][:30])}...")

# Test encoding/decoding
test_text = "Hello World 123!"
encoded = encoder.encode(test_text)
print(f"\n📊 Test encode/decode:")
print(f"   Text: '{test_text}'")
print(f"   Encoded: {encoded}")

✅ Loaded charset from: D:\d121221080_HTR_ta\Tugas_Akhir\synthetic_dataset\synthetic_lines_1M\synthetic_lines_1M\charset.json
📝 CHARACTER ENCODER
   Total classes: 83 (including BLANK)
   Letters: 59
   Digits: 10
   Punctuation: 12
   Has space: True

   Sample chars:  !"&'(),-.0123456789...

📊 Test encode/decode:
   Text: 'Hello World 123!'
   Encoded: [31, 54, 61, 61, 64, 1, 46, 64, 67, 61, 53, 1, 12, 13, 14, 2]


## Section 4: Dataset & DataLoader

In [8]:
class HTRDataset(Dataset):
    """
    Dataset for Handwritten Text Recognition
    
    Loads images and text labels from CSV file.
    Supports data augmentation for training.
    """
    
    def __init__(
        self,
        data_df: pd.DataFrame,
        encoder: CharacterEncoder,
        image_height: int = 64,
        image_width: int = 512,
        augment: bool = False,
        base_path: str = ""
    ):
        self.data = data_df.reset_index(drop=True)
        self.encoder = encoder
        self.image_height = image_height
        self.image_width = image_width
        self.base_path = base_path
        
        # Augmentation pipeline (untuk training)
        if augment:
            self.transform = A.Compose([
                # Geometric augmentation
                A.Affine(
                    rotate=(-2, 2),        # Rotasi kecil
                    shear=(-3, 3),         # Shear sedikit
                    scale=(0.95, 1.05),    # Scale sedikit
                    p=0.5
                ),
                # Blur augmentation
                A.OneOf([
                    A.GaussianBlur(blur_limit=(1, 3)),
                    A.MotionBlur(blur_limit=3),
                ], p=0.2),
                # Brightness/Contrast
                A.RandomBrightnessContrast(
                    brightness_limit=0.1,
                    contrast_limit=0.1,
                    p=0.3
                ),
                # Noise - compatible with different albumentations versions
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None
        
        # Statistics
        self.load_failures = 0
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, int, str]:
        row = self.data.iloc[idx]
        
        # Get image path
        img_path = row['path']
        
        # Handle relative paths
        if self.base_path and not os.path.isabs(img_path):
            img_path = os.path.join(self.base_path, img_path)
        
        # Load image as grayscale
        try:
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError(f"Cannot load: {img_path}")
        except Exception as e:
            # Return blank image if loading fails
            self.load_failures += 1
            img = np.ones((self.image_height, self.image_width), dtype=np.uint8) * 255
        
        # Get text
        text = str(row['text'])
        
        # Resize maintaining aspect ratio
        h, w = img.shape[:2]
        new_h = self.image_height
        new_w = int(w * new_h / h)
        
        # Limit width to max
        if new_w > self.image_width:
            new_w = self.image_width
        
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        # Pad to fixed width (right padding with white)
        if new_w < self.image_width:
            pad_w = self.image_width - new_w
            img = np.pad(img, ((0, 0), (0, pad_w)), mode='constant', constant_values=255)
        
        # Apply augmentation
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        # Normalize to [0, 1] and invert (dark text on light background)
        img = img.astype(np.float32) / 255.0
        img = 1.0 - img  # Invert: text becomes high values (white = 0, black = 1)
        
        # Convert to tensor: (1, H, W) - single channel
        img_tensor = torch.FloatTensor(img).unsqueeze(0)
        
        # Encode text to indices
        encoded = self.encoder.encode(text)
        target = torch.LongTensor(encoded)
        target_length = len(encoded)
        
        return img_tensor, target, target_length, text

print("✅ HTRDataset class ready!")
print(f"   Supports augmentation: rotation, shear, blur, brightness, noise")

✅ HTRDataset class ready!
   Supports augmentation: rotation, shear, blur, brightness, noise


In [9]:
def collate_fn(batch):
    """Custom collate function for variable length targets"""
    images, targets, target_lengths, texts = zip(*batch)
    
    # Stack images
    images = torch.stack(images, 0)  # (B, 1, H, W)
    
    # Pad targets
    target_lengths = torch.LongTensor(target_lengths)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)
    
    return images, targets_padded, target_lengths, texts

print("✅ Collate function ready!")

✅ Collate function ready!


## Section 5: CRNN Model Architecture

In [10]:
class CNNBackbone(nn.Module):
    """CNN feature extractor (VGG-like)"""
    
    def __init__(self, input_channels: int = 1, output_channels: int = 512):
        super().__init__()
        
        self.cnn = nn.Sequential(
            # Block 1: 64 -> 64
            nn.Conv2d(input_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # H/2, W/2
            
            # Block 2: 64 -> 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # H/4, W/4
            
            # Block 3: 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),  # H/8, W/4
            
            # Block 4: 256 -> 512
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),  # H/16, W/4
            
            # Block 5: 512 -> output_channels
            nn.Conv2d(512, output_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, None)),  # H=1, W preserved
        )
        
    def forward(self, x):
        # x: (B, 1, H, W)
        conv = self.cnn(x)  # (B, C, 1, W')
        conv = conv.squeeze(2)  # (B, C, W')
        conv = conv.permute(0, 2, 1)  # (B, W', C) - sequence format
        return conv

print("✅ CNNBackbone ready!")

✅ CNNBackbone ready!


In [11]:
class CRNN(nn.Module):
    """CRNN model for Handwritten Text Recognition"""
    
    def __init__(
        self,
        num_classes: int,
        cnn_output_channels: int = 512,
        rnn_hidden_size: int = 256,
        rnn_num_layers: int = 2,
        rnn_dropout: float = 0.3,
        bidirectional: bool = True
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.rnn_hidden_size = rnn_hidden_size
        self.bidirectional = bidirectional
        
        # CNN backbone
        self.cnn = CNNBackbone(
            input_channels=1,
            output_channels=cnn_output_channels
        )
        
        # Bidirectional LSTM
        self.rnn = nn.LSTM(
            input_size=cnn_output_channels,
            hidden_size=rnn_hidden_size,
            num_layers=rnn_num_layers,
            batch_first=True,
            dropout=rnn_dropout if rnn_num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # Output projection
        rnn_output_size = rnn_hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(rnn_output_size, num_classes)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize model weights"""
        for name, param in self.named_parameters():
            if 'weight' in name:
                if 'lstm' in name.lower() or 'rnn' in name.lower():
                    nn.init.orthogonal_(param)
                elif len(param.shape) >= 2:
                    nn.init.kaiming_normal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input images (B, 1, H, W)
            
        Returns:
            log_probs: Log probabilities (T, B, num_classes) for CTC
        """
        # CNN features
        conv = self.cnn(x)  # (B, T, C)
        
        # RNN
        rnn_out, _ = self.rnn(conv)  # (B, T, H*2)
        
        # FC projection
        output = self.fc(rnn_out)  # (B, T, num_classes)
        
        # Permute for CTC: (T, B, C)
        output = output.permute(1, 0, 2)
        
        # Log softmax for CTC loss
        log_probs = F.log_softmax(output, dim=2)
        
        return log_probs
    
    def get_sequence_length(self, input_width: int) -> int:
        """Calculate output sequence length given input width"""
        # After CNN: W/4 (due to pooling)
        return input_width // 4

print("✅ CRNN model ready!")

✅ CRNN model ready!


In [12]:
# Test model
print("Testing CRNN model...")

test_model = CRNN(
    num_classes=encoder.num_classes,
    cnn_output_channels=config.cnn_output_channels,
    rnn_hidden_size=config.rnn_hidden_size,
    rnn_num_layers=config.rnn_num_layers,
    rnn_dropout=config.rnn_dropout,
    bidirectional=config.bidirectional
).to(device)

# Test forward pass
dummy_input = torch.randn(4, 1, 64, 256).to(device)
dummy_output = test_model(dummy_input)

print(f"\n✅ Model test passed!")
print(f"   Input shape: {dummy_input.shape}")
print(f"   Output shape: {dummy_output.shape}")  # (T, B, num_classes)
print(f"   Sequence length: {dummy_output.shape[0]}")

# Count parameters
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

del test_model, dummy_input, dummy_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Testing CRNN model...

✅ Model test passed!
   Input shape: torch.Size([4, 1, 64, 256])
   Output shape: torch.Size([64, 4, 83])
   Sequence length: 64
   Total parameters: 10,060,499
   Trainable parameters: 10,060,499


## Section 6: Metrics (CER & WER)

In [13]:
def calculate_cer(predictions: List[str], targets: List[str]) -> float:
    """
    Calculate Character Error Rate (CER)
    
    CER = (Substitutions + Insertions + Deletions) / Total Characters
    Uses rapidfuzz.distance.Levenshtein for Python 3.13 compatibility
    """
    total_chars = 0
    total_errors = 0
    
    for pred, target in zip(predictions, targets):
        errors = Levenshtein.distance(pred, target)
        total_errors += errors
        total_chars += len(target)
    
    if total_chars == 0:
        return 0.0
    
    return total_errors / total_chars


def calculate_wer(predictions: List[str], targets: List[str]) -> float:
    """
    Calculate Word Error Rate (WER)
    
    WER = (Substitutions + Insertions + Deletions) / Total Words
    Uses rapidfuzz.distance.Levenshtein for Python 3.13 compatibility
    """
    total_words = 0
    total_errors = 0
    
    for pred, target in zip(predictions, targets):
        pred_words = pred.split()
        target_words = target.split()
        
        # Levenshtein.distance works with sequences (lists)
        errors = Levenshtein.distance(pred_words, target_words)
        total_errors += errors
        total_words += len(target_words)
    
    if total_words == 0:
        return 0.0
    
    return total_errors / total_words


def decode_predictions(log_probs: torch.Tensor, encoder: 'CharacterEncoder') -> List[str]:
    """
    Decode CTC output to text using greedy decoding
    
    Args:
        log_probs: (T, B, num_classes) - log probabilities from model
        encoder: Character encoder for decoding
        
    Returns:
        List of decoded strings
    """
    # Greedy decoding: take argmax at each timestep
    # log_probs: (T, B, C)
    predictions = log_probs.argmax(dim=2)  # (T, B)
    predictions = predictions.permute(1, 0)  # (B, T)
    
    decoded = encoder.decode_batch(predictions)
    return decoded


print("✅ Metrics functions ready!")
print("   Using rapidfuzz.distance.Levenshtein (Python 3.13 compatible)")

# Test metrics
test_pred = ["hello world", "testing one two"]
test_target = ["hello world", "testing 1 2 3"]
print(f"   Sample CER: {calculate_cer(test_pred, test_target):.4f}")
print(f"   Sample WER: {calculate_wer(test_pred, test_target):.4f}")

✅ Metrics functions ready!
   Using rapidfuzz.distance.Levenshtein (Python 3.13 compatible)
   Sample CER: 0.2500
   Sample WER: 0.5000


## Section 7: Training Functions

In [14]:
# ============================================
# TRAINER CLASS
# ============================================

import time
from datetime import datetime
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

class HTRTrainer:
    """Complete training pipeline for HTR model"""
    
    def __init__(
        self,
        model: nn.Module,
        encoder: CharacterEncoder,
        config: TrainingConfig,
        device: torch.device,
        log_dir: Path,
        checkpoint_dir: Path
    ):
        self.model = model.to(device)
        self.encoder = encoder
        self.config = config
        self.device = device
        self.log_dir = Path(log_dir)
        self.checkpoint_dir = Path(checkpoint_dir)
        
        # Create directories
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Training state
        self.current_epoch = 0
        self.global_step = 0
        self.best_val_cer = float('inf')
        self.best_epoch = 0
        self.num_epochs = 10
        
        # History for plotting
        self.history = {
            'epoch': [],
            'train_loss': [],
            'val_loss': [],
            'val_cer': [],
            'val_wer': [],
            'lr': []
        }
        
        # CTC Loss
        self.ctc_loss = nn.CTCLoss(blank=encoder.blank_idx, reduction='mean', zero_infinity=True)
        
        print(f"✅ Trainer initialized")
        print(f"   Log dir: {self.log_dir}")
        print(f"   Checkpoint dir: {self.checkpoint_dir}")
    
    def train_epoch(self, train_loader: DataLoader, optimizer) -> Tuple[float, float]:
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        num_batches = 0
        start_time = time.time()
        
        pbar = tqdm(train_loader, desc="Training", leave=False)
        for batch in pbar:
            images, targets, target_lengths, texts = batch
            images = images.to(self.device)
            
            # Forward pass
            log_probs = self.model(images)
            
            # CTC loss
            T, B, C = log_probs.shape
            input_lengths = torch.full((B,), T, dtype=torch.long, device=self.device)
            
            # Flatten targets for CTC
            target_flat = torch.cat([targets[i, :target_lengths[i]] for i in range(B)])
            
            loss = self.ctc_loss(log_probs, target_flat, input_lengths, target_lengths)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.grad_clip)
            
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            self.global_step += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        epoch_time = time.time() - start_time
        avg_loss = total_loss / max(num_batches, 1)
        return avg_loss, epoch_time
    
    @torch.no_grad()
    def validate(self, val_loader: DataLoader) -> Tuple[float, float, float, List]:
        """Validate and compute metrics"""
        self.model.eval()
        total_loss = 0
        num_batches = 0
        all_predictions = []
        all_targets = []
        sample_results = []
        
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            images, targets, target_lengths, texts = batch
            images = images.to(self.device)
            
            # Forward pass
            log_probs = self.model(images)
            T, B, C = log_probs.shape
            input_lengths = torch.full((B,), T, dtype=torch.long, device=self.device)
            
            # Flatten targets for CTC
            target_flat = torch.cat([targets[i, :target_lengths[i]] for i in range(B)])
            
            loss = self.ctc_loss(log_probs, target_flat, input_lengths, target_lengths)
            total_loss += loss.item()
            num_batches += 1
            
            # Decode predictions
            pred_indices = log_probs.argmax(dim=2).permute(1, 0).cpu().numpy()
            
            for i, (pred, true_text) in enumerate(zip(pred_indices, texts)):
                # CTC decode
                decoded = []
                prev = -1
                for idx in pred:
                    if idx != self.encoder.blank_idx and idx != prev:
                        decoded.append(idx)
                    prev = idx
                
                pred_text = self.encoder.decode(decoded)
                all_predictions.append(pred_text)
                all_targets.append(true_text)
                
                if len(sample_results) < 5:
                    sample_results.append({
                        'prediction': pred_text,
                        'ground_truth': true_text
                    })
        
        avg_loss = total_loss / max(num_batches, 1)
        cer = calculate_cer(all_predictions, all_targets)
        wer = calculate_wer(all_predictions, all_targets)
        
        return avg_loss, cer, wer, sample_results
    
    def fit(
        self,
        train_loader: DataLoader,
        val_loader: DataLoader,
        num_epochs: int = 10,
        learning_rate: float = 1e-3,
        checkpoint_prefix: str = "model"
    ) -> Dict:
        """Full training loop with logging and checkpointing"""
        
        self.num_epochs = num_epochs
        
        # Optimizer
        optimizer = AdamW(
            self.model.parameters(),
            lr=learning_rate,
            weight_decay=self.config.weight_decay
        )
        
        # Scheduler
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=self.config.scheduler_factor,
            patience=self.config.scheduler_patience,
            min_lr=self.config.min_lr
        )
        
        print("\n" + "=" * 70)
        print("🚀 STARTING TRAINING")
        print("=" * 70)
        print(f"   Epochs: {num_epochs}")
        print(f"   Learning rate: {learning_rate}")
        print(f"   Device: {self.device}")
        print(f"   Checkpoints: {self.checkpoint_dir}")
        print("=" * 70 + "\n")
        
        training_start_time = datetime.now()
        
        for epoch in range(num_epochs):
            self.current_epoch = epoch
            current_lr = optimizer.param_groups[0]['lr']
            
            print(f"\n{'='*60}")
            print(f"Epoch {epoch+1}/{num_epochs} | LR: {current_lr:.2e}")
            print(f"{'='*60}")
            
            # Train
            train_loss, epoch_time = self.train_epoch(train_loader, optimizer)
            
            # Validate
            val_loss, val_cer, val_wer, samples = self.validate(val_loader)
            
            # Update scheduler
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_loss)
            new_lr = optimizer.param_groups[0]['lr']
            
            if new_lr < old_lr:
                print(f"   📉 Learning rate reduced: {old_lr:.2e} → {new_lr:.2e}")
            
            # Log history
            self.history['epoch'].append(epoch + 1)
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_cer'].append(val_cer)
            self.history['val_wer'].append(val_wer)
            self.history['lr'].append(current_lr)
            
            # Print epoch summary
            print(f"\n📊 Epoch {epoch+1} Summary:")
            print(f"   Train Loss: {train_loss:.4f}")
            print(f"   Val Loss:   {val_loss:.4f}")
            print(f"   Val CER:    {val_cer:.4f} ({val_cer*100:.2f}%)")
            print(f"   Val WER:    {val_wer:.4f} ({val_wer*100:.2f}%)")
            print(f"   Time:       {epoch_time:.1f}s")
            
            # Show samples
            print(f"\n📝 Sample Predictions:")
            for i, sample in enumerate(samples[:3]):
                print(f"   [{i+1}] GT:   '{sample['ground_truth']}'")
                print(f"       Pred: '{sample['prediction']}'")
            
            # Check if best model
            is_best = val_cer < self.best_val_cer
            if is_best:
                self.best_val_cer = val_cer
                self.best_epoch = epoch + 1
                print(f"\n   🏆 New best model! CER: {val_cer:.4f}")
            
            # Save checkpoints
            self.save_checkpoint(
                optimizer, scheduler, 
                f"{checkpoint_prefix}_epoch_{epoch+1}.pt",
                is_best=is_best, is_final=False
            )
            if is_best:
                self.save_checkpoint(
                    optimizer, scheduler,
                    f"{checkpoint_prefix}_best.pt",
                    is_best=True, is_final=False
                )
            
            # Save history
            self.save_history()
            
            # Early stopping
            epochs_without_improvement = epoch - (self.best_epoch - 1)
            if epochs_without_improvement >= self.config.early_stop_patience:
                print(f"\n⚠️ Early stopping triggered after {epochs_without_improvement} epochs without improvement")
                break
        
        # Save final model
        self.save_checkpoint(
            optimizer, scheduler,
            f"{checkpoint_prefix}_final.pt",
            is_best=False, is_final=True
        )
        
        training_time = datetime.now() - training_start_time
        print(f"\n✅ Training completed!")
        print(f"   Total time: {training_time}")
        print(f"   Best CER: {self.best_val_cer:.4f} at epoch {self.best_epoch}")
        
        return {
            'best_cer': self.best_val_cer,
            'best_epoch': self.best_epoch,
            'history': self.history
        }
    
    def save_checkpoint(
        self, optimizer, scheduler, filename: str,
        is_best: bool = False, is_final: bool = False
    ):
        """Save training checkpoint"""
        path = self.checkpoint_dir / filename
        checkpoint = {
            'epoch': self.current_epoch + 1,
            'global_step': self.global_step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_cer': self.best_val_cer,
            'best_epoch': self.best_epoch,
            'history': self.history,
            'encoder': self.encoder.char_to_idx,
            'is_best': is_best,
            'is_final': is_final,
        }
        torch.save(checkpoint, path)
        print(f"   💾 Saved: {path.name}")
    
    def save_history(self):
        """Save training history to JSON"""
        history_path = self.log_dir / "training_history.json"
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load_checkpoint(self, path: str) -> Dict:
        """Load checkpoint and restore state"""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.current_epoch = checkpoint['epoch']
        self.global_step = checkpoint['global_step']
        self.best_val_cer = checkpoint['best_val_cer']
        self.best_epoch = checkpoint['best_epoch']
        self.history = checkpoint['history']
        print(f"✅ Loaded checkpoint from epoch {checkpoint['epoch']}")
        return checkpoint

# Alias
Trainer = HTRTrainer

print("✅ Trainer class ready!")

✅ Trainer class ready!


## Section 8: Load Data

In [15]:
# ============================================
# LOAD SYNTHETIC DATASET (Phase 1)
# ============================================

print("=" * 60)
print("📂 LOADING SYNTHETIC DATASET (Phase 1: Pre-training)")
print("=" * 60)

# Load synthetic data CSVs
synth_train_df = pd.read_csv(SYNTHETIC_DATA_DIR / "train_synthetic.csv")
synth_val_df = pd.read_csv(SYNTHETIC_DATA_DIR / "val_synthetic.csv")

# Fix paths - make sure they're absolute
def fix_local_paths(df, base_dir):
    """Ensure paths are absolute and valid for local system"""
    if 'path' in df.columns:
        # Check if paths are already absolute
        sample_path = df.iloc[0]['path']
        if not os.path.isabs(sample_path):
            df['path'] = df['path'].apply(lambda x: str(base_dir / x))
    return df

synth_train_df = fix_local_paths(synth_train_df, SYNTHETIC_DATA_DIR)
synth_val_df = fix_local_paths(synth_val_df, SYNTHETIC_DATA_DIR)

print(f"\n📊 Synthetic Dataset:")
print(f"   Train: {len(synth_train_df):,} samples")
print(f"   Val: {len(synth_val_df):,} samples")
print(f"   Total: {len(synth_train_df) + len(synth_val_df):,} samples")

# ============================================
# ⚠️ VERIFIKASI DATA - PENTING!
# ============================================
print(f"\n🔍 VERIFYING DATA (checking first 100 samples)...")

valid_count = 0
invalid_paths = []
empty_texts = []

for idx in range(min(100, len(synth_train_df))):
    row = synth_train_df.iloc[idx]
    path = row['path']
    text = str(row['text'])
    
    # Check path exists
    if not os.path.exists(path):
        invalid_paths.append(path)
    else:
        valid_count += 1
    
    # Check text not empty
    if len(text.strip()) == 0:
        empty_texts.append(idx)

print(f"\n📊 Verification Results:")
print(f"   Valid paths: {valid_count}/100")
print(f"   Invalid paths: {len(invalid_paths)}")
print(f"   Empty texts: {len(empty_texts)}")

if invalid_paths:
    print(f"\n❌ MASALAH: Ada {len(invalid_paths)} path tidak valid!")
    print(f"   Contoh path: {invalid_paths[0]}")
    print(f"\n   SOLUSI:")
    print(f"   1. Pastikan folder synthetic data ada di: {SYNTHETIC_DATA_DIR}")
    print(f"   2. Pastikan folder 'images/' ada di dalam synthetic_lines_1M")
else:
    print(f"\n✅ Semua path valid! Data siap untuk training.")

# Show sample
print(f"\n📝 Sample Data:")
sample = synth_train_df.iloc[0]
print(f"   Path: {sample['path']}")
print(f"   Text: '{sample['text']}'")
print(f"   Path exists: {os.path.exists(sample['path'])}")

# Test load image
sample_path = synth_train_df.iloc[0]['path']
if os.path.exists(sample_path):
    test_img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
    if test_img is not None:
        print(f"   Image shape: {test_img.shape}")
        print(f"   ✅ Image loads correctly!")
    else:
        print(f"   ❌ Image failed to load (cv2 returned None)")
else:
    print(f"   ❌ Path does not exist!")

📂 LOADING SYNTHETIC DATASET (Phase 1: Pre-training)

📊 Synthetic Dataset:
   Train: 900,000 samples
   Val: 50,000 samples
   Total: 950,000 samples

🔍 VERIFYING DATA (checking first 100 samples)...

📊 Verification Results:
   Valid paths: 0/100
   Invalid paths: 100
   Empty texts: 0

❌ MASALAH: Ada 100 path tidak valid!
   Contoh path: D:\Dokumen Kuliah\TA\Kode Kaggle\synthetic dataset\synthetic_lines_1M\batch_0197\synth_00987231.png

   SOLUSI:
   1. Pastikan folder synthetic data ada di: D:\d121221080_HTR_ta\Tugas_Akhir\synthetic_dataset\synthetic_lines_1M\synthetic_lines_1M
   2. Pastikan folder 'images/' ada di dalam synthetic_lines_1M

📝 Sample Data:
   Path: D:\Dokumen Kuliah\TA\Kode Kaggle\synthetic dataset\synthetic_lines_1M\batch_0197\synth_00987231.png
   Text: 'have arisen since the post - war era :'
   Path exists: False
   ❌ Path does not exist!


In [16]:
# ============================================
# LOAD SYNTHETIC DATASET (Phase 1) - FIXED
# ============================================

from pathlib import Path  # Pastikan library ini terpanggil

print("=" * 60)
print("📂 LOADING SYNTHETIC DATASET (Phase 1: Pre-training)")
print("=" * 60)

# Load synthetic data CSVs
synth_train_df = pd.read_csv(SYNTHETIC_DATA_DIR / "train_synthetic.csv")
synth_val_df = pd.read_csv(SYNTHETIC_DATA_DIR / "val_synthetic.csv")

# --- BAGIAN YANG DIPERBAIKI ---
def fix_local_paths(df, base_dir):
    """
    Memaksa update path agar sesuai dengan lokasi folder saat ini.
    Mengambil 'batch_xxxx/filename.png' dari path lama dan menempelkannya ke base_dir baru.
    """
    if 'path' in df.columns:
        def correct_path(old_path):
            # Ubah path string jadi objek Path
            p = Path(old_path)
            # Ambil 2 bagian terakhir (contoh: 'batch_0197' dan 'synth_00987231.png')
            # Ini akan membuang 'D:\Dokumen Kuliah\...' yang salah
            relative_path = Path(p.parts[-2]) / p.parts[-1]
            
            # Gabungkan dengan folder yang benar di komputer ini
            return str(base_dir / relative_path)

        # Terapkan perbaikan ke semua baris
        df['path'] = df['path'].apply(correct_path)
    return df
# ------------------------------

synth_train_df = fix_local_paths(synth_train_df, SYNTHETIC_DATA_DIR)
synth_val_df = fix_local_paths(synth_val_df, SYNTHETIC_DATA_DIR)

print(f"\n📊 Synthetic Dataset:")
print(f"   Train: {len(synth_train_df):,} samples")
print(f"   Val: {len(synth_val_df):,} samples")
print(f"   Total: {len(synth_train_df) + len(synth_val_df):,} samples")

# ============================================
# ⚠️ VERIFIKASI DATA
# ============================================
print(f"\n🔍 VERIFYING DATA (checking first 100 samples)...")

valid_count = 0
invalid_paths = []

for idx in range(min(100, len(synth_train_df))):
    row = synth_train_df.iloc[idx]
    if not os.path.exists(row['path']):
        invalid_paths.append(row['path'])
    else:
        valid_count += 1

print(f"\n📊 Verification Results:")
print(f"   Valid paths: {valid_count}/100")

if invalid_paths:
    print(f"\n❌ MASIH ADA ERROR PADA PATH:")
    print(f"   Target Path (Baru): {invalid_paths[0]}")
    print(f"   Pastikan folder 'batch_xxxx' benar-benar ada di dalam:")
    print(f"   {SYNTHETIC_DATA_DIR}")
else:
    print(f"\n✅ SUCCESS! Semua path valid.")

📂 LOADING SYNTHETIC DATASET (Phase 1: Pre-training)

📊 Synthetic Dataset:
   Train: 900,000 samples
   Val: 50,000 samples
   Total: 950,000 samples

🔍 VERIFYING DATA (checking first 100 samples)...

📊 Verification Results:
   Valid paths: 100/100

✅ SUCCESS! Semua path valid.


In [17]:
# ============================================
# LOAD IAM DATASET FROM HUGGING FACE (Phase 2)
# ============================================

print("=" * 60)
print("📂 LOADING IAM DATASET")
print("=" * 60)

IAM_LOADED = False

# Option 1: Load from local directory if available
if IAM_DATA_DIR and Path(IAM_DATA_DIR).exists():
    print(f"\n📁 Loading IAM from local directory: {IAM_DATA_DIR}")
    # Implement local loading logic here if you have IAM locally
    # iam_train_df = pd.read_csv(IAM_DATA_DIR / "train.csv")
    # etc.
    pass

# Option 2: Download from Hugging Face
if not IAM_LOADED:
    try:
        from datasets import load_dataset
        
        print("\n📥 Downloading IAM Lines from Hugging Face...")
        print("   This may take a few minutes on first run...")
        
        # Load dataset
        iam_dataset = load_dataset("Teklia/IAM-line", trust_remote_code=True)
        
        print(f"\n✅ IAM Dataset loaded!")
        print(f"   Train: {len(iam_dataset['train']):,} samples")
        print(f"   Validation: {len(iam_dataset['validation']):,} samples") 
        print(f"   Test: {len(iam_dataset['test']):,} samples")
        
        # Convert to DataFrame format
        IAM_IMAGES_DIR = OUTPUT_DIR / "iam_images"
        
        def hf_to_dataframe(hf_split, split_name):
            """Convert Hugging Face dataset split to DataFrame"""
            data = []
            img_dir = IAM_IMAGES_DIR / split_name
            img_dir.mkdir(parents=True, exist_ok=True)
            
            for idx, item in enumerate(tqdm(hf_split, desc=f"Processing {split_name}")):
                img_path = img_dir / f"{idx:06d}.png"
                
                # Save PIL image
                item['image'].save(str(img_path))
                
                data.append({
                    'path': str(img_path),
                    'text': item['text'],
                    'line_id': f"{split_name}_{idx:06d}"
                })
            
            return pd.DataFrame(data)
        
        print("\n📦 Converting to DataFrame format...")
        print(f"   (Saving images to {IAM_IMAGES_DIR})")
        
        iam_train_df = hf_to_dataframe(iam_dataset['train'], 'train')
        iam_val_df = hf_to_dataframe(iam_dataset['validation'], 'val')
        iam_test_df = hf_to_dataframe(iam_dataset['test'], 'test')
        
        print(f"\n✅ Conversion complete!")
        print(f"   Train: {len(iam_train_df):,} lines")
        print(f"   Val: {len(iam_val_df):,} lines")
        print(f"   Test: {len(iam_test_df):,} lines")
        
        IAM_LOADED = True
        
    except ImportError:
        print("\n⚠️ datasets library not installed. Run: pip install datasets")
    except Exception as e:
        print(f"\n❌ Failed to load from Hugging Face: {e}")
        
if not IAM_LOADED:
    print("\n⚠️ IAM dataset not loaded. Phase 2 (fine-tuning) will be skipped.")

📂 LOADING IAM DATASET


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Teklia/IAM-line' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📥 Downloading IAM Lines from Hugging Face...
   This may take a few minutes on first run...


'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)' thrown while requesting HEAD https://huggingface.co/datasets/Teklia/IAM-line/resolve/fbdad97500ce54635c0d1ba306bf535cb40656cf/IAM-line.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since Teklia/IAM-line couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\User\.cache\huggingface\datasets\Teklia___iam-line\default\0.0.0\fbdad97500ce54635c0d1ba306bf535cb40656cf (last modified on Thu Feb  5 14:12:21 2026).



✅ IAM Dataset loaded!
   Train: 6,482 samples
   Validation: 976 samples
   Test: 2,915 samples

📦 Converting to DataFrame format...
   (Saving images to D:\d121221080_HTR_ta\Tugas_Akhir\training_output\iam_images)


Processing test: 100%|██████████| 2915/2915 [00:14<00:00, 196.45it/s]


✅ Conversion complete!
   Train: 6,482 lines
   Val: 976 lines
   Test: 2,915 lines


## Section 9: Phase 1 - Pre-training on Synthetic Data

In [18]:
# ============================================
# PHASE 1: PRE-TRAINING ON SYNTHETIC DATA
# ============================================

print("\n" + "=" * 70)
print("🚀 PHASE 1: PRE-TRAINING ON SYNTHETIC DATA")
print("=" * 70)
print(f"   Dataset: {len(synth_train_df):,} train, {len(synth_val_df):,} val")
print(f"   Epochs: {config.pretrain_epochs}")
print(f"   Batch size: {config.pretrain_batch_size}")
print(f"   Learning rate: {config.pretrain_lr}")
print(f"   num_workers: {config.num_workers} (0 = no multiprocessing)")
print("=" * 70)

# Create datasets
synth_train_dataset = HTRDataset(
    synth_train_df,
    encoder,
    image_height=config.image_height,
    image_width=config.image_width,
    augment=config.augment_train,
    base_path=""
)

synth_val_dataset = HTRDataset(
    synth_val_df,
    encoder,
    image_height=config.image_height,
    image_width=config.image_width,
    augment=False,
    base_path=""
)

# ⚠️ PENTING: Test dataset sebelum training
print("\n🔍 Testing dataset...")
try:
    test_item = synth_train_dataset[0]
    img_tensor, target, target_len, text = test_item
    print(f"   ✅ Dataset works!")
    print(f"   Image shape: {img_tensor.shape}")
    print(f"   Target length: {target_len}")
    print(f"   Text: '{text}'")
    print(f"   Encoded: {target[:10].tolist()}...")
except Exception as e:
    print(f"   ❌ Dataset error: {e}")
    raise

# Create dataloaders
# ⚠️ PENTING: num_workers=0 untuk Windows!
synth_train_loader = DataLoader(
    synth_train_dataset,
    batch_size=config.pretrain_batch_size,
    shuffle=True,
    num_workers=0,  # ⚠️ HARUS 0 untuk Windows!
    pin_memory=config.pin_memory,
    collate_fn=collate_fn,
    drop_last=True
)

synth_val_loader = DataLoader(
    synth_val_dataset,
    batch_size=config.pretrain_batch_size,
    shuffle=False,
    num_workers=0,  # ⚠️ HARUS 0 untuk Windows!
    pin_memory=config.pin_memory,
    collate_fn=collate_fn
)

print(f"\n📦 DataLoaders created (num_workers=0):")
print(f"   Train: {len(synth_train_loader):,} batches")
print(f"   Val: {len(synth_val_loader):,} batches")

# Test dataloader
print("\n🔍 Testing dataloader...")
try:
    test_batch = next(iter(synth_train_loader))
    images, targets, target_lengths, texts = test_batch
    print(f"   ✅ DataLoader works!")
    print(f"   Batch images shape: {images.shape}")
    print(f"   Sample text: '{texts[0]}'")
except Exception as e:
    print(f"   ❌ DataLoader error: {e}")
    raise

# Create model
model = CRNN(
    num_classes=encoder.num_classes,
    cnn_output_channels=config.cnn_output_channels,
    rnn_hidden_size=config.rnn_hidden_size,
    rnn_num_layers=config.rnn_num_layers,
    rnn_dropout=config.rnn_dropout,
    bidirectional=config.bidirectional
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\n🏗️  Model created: {total_params:,} parameters")

# Create trainer
pretrain_trainer = HTRTrainer(
    model=model,
    encoder=encoder,
    config=config,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    log_dir=RESULTS_DIR
)

# Pre-train!
print("\n🔄 Starting pre-training...")
print("   (Progress bar should update every batch)")
pretrain_history = pretrain_trainer.fit(
    synth_train_loader,
    synth_val_loader,
    num_epochs=config.pretrain_epochs,
    learning_rate=config.pretrain_lr,
    checkpoint_prefix="pretrain"
)

print("\n✅ Phase 1 (Pre-training) complete!")


🚀 PHASE 1: PRE-TRAINING ON SYNTHETIC DATA
   Dataset: 900,000 train, 50,000 val
   Epochs: 3
   Batch size: 64
   Learning rate: 0.001
   num_workers: 0 (0 = no multiprocessing)

🔍 Testing dataset...
   ✅ Dataset works!
   Image shape: torch.Size([1, 64, 512])
   Target length: 38
   Text: 'have arisen since the post - war era :'
   Encoded: [57, 50, 71, 54, 1, 50, 67, 58, 68, 54]...

📦 DataLoaders created (num_workers=0):
   Train: 14,062 batches
   Val: 782 batches

🔍 Testing dataloader...
   ✅ DataLoader works!
   Batch images shape: torch.Size([64, 1, 64, 512])
   Sample text: 'of a large variety of'

🏗️  Model created: 10,060,499 parameters
✅ Trainer initialized
   Log dir: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\results
   Checkpoint dir: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\checkpoints

🔄 Starting pre-training...
   (Progress bar should update every batch)

🚀 STARTING TRAINING
   Epochs: 3
   Learning rate: 0.001
   Device: cuda
   Checkpoints: D:\d12122108


📊 Epoch 1 Summary:
   Train Loss: 0.0796
   Val Loss:   0.0023
   Val CER:    0.0182 (1.82%)
   Val WER:    0.0887 (8.87%)
   Time:       10314.9s

📝 Sample Predictions:
   [1] GT:   'Monagan of Connecticut'
       Pred: 'Monagan of Conecticut'
   [2] GT:   'coffee '' ? ? No , thanks '' .'
       Pred: 'cofe ' ? ? No , thanks "' .'
   [3] GT:   'Matt Hardy with help from Ivory , but'
       Pred: 'Mat Hardy with help from Ivory , but'

   🏆 New best model! CER: 0.0182
   💾 Saved: pretrain_epoch_1.pt
   💾 Saved: pretrain_best.pt

Epoch 2/3 | LR: 1.00e-03



📊 Epoch 2 Summary:
   Train Loss: 0.0073
   Val Loss:   0.0009
   Val CER:    0.0179 (1.79%)
   Val WER:    0.0879 (8.79%)
   Time:       10276.4s

📝 Sample Predictions:
   [1] GT:   'Monagan of Connecticut'
       Pred: 'Monagan of Conecticut'
   [2] GT:   'coffee '' ? ? No , thanks '' .'
       Pred: 'cofe ' ? ? No , thanks ' .'
   [3] GT:   'Matt Hardy with help from Ivory , but'
       Pred: 'Mat Hardy with help from Ivory , but'

   🏆 New best model! CER: 0.0179
   💾 Saved: pretrain_epoch_2.pt
   💾 Saved: pretrain_best.pt

Epoch 3/3 | LR: 1.00e-03



📊 Epoch 3 Summary:
   Train Loss: 0.0052
   Val Loss:   0.0006
   Val CER:    0.0178 (1.78%)
   Val WER:    0.0875 (8.75%)
   Time:       10270.9s

📝 Sample Predictions:
   [1] GT:   'Monagan of Connecticut'
       Pred: 'Monagan of Conecticut'
   [2] GT:   'coffee '' ? ? No , thanks '' .'
       Pred: 'cofe ' ? ? No , thanks ' .'
   [3] GT:   'Matt Hardy with help from Ivory , but'
       Pred: 'Mat Hardy with help from Ivory , but'

   🏆 New best model! CER: 0.0178
   💾 Saved: pretrain_epoch_3.pt
   💾 Saved: pretrain_best.pt
   💾 Saved: pretrain_final.pt

✅ Training completed!
   Total time: 8:52:28.866616
   Best CER: 0.0178 at epoch 3

✅ Phase 1 (Pre-training) complete!


In [19]:
# ============================================
# 💾 SAVE PRE-TRAINED MODEL (untuk eksperimen)
# ============================================

print("\n" + "=" * 60)
print("💾 SAVING PRE-TRAINED MODEL")
print("=" * 60)

# Directory untuk menyimpan model eksperimen
MODELS_DIR = OUTPUT_DIR / "saved_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save format 1: Full checkpoint (untuk resume training)
pretrain_full_path = MODELS_DIR / "pretrain_synthetic_full.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'encoder_charset': encoder.char_to_idx,
    'config': {
        'num_classes': encoder.num_classes,
        'cnn_output_channels': config.cnn_output_channels,
        'rnn_hidden_size': config.rnn_hidden_size,
        'rnn_num_layers': config.rnn_num_layers,
        'rnn_dropout': config.rnn_dropout,
        'bidirectional': config.bidirectional,
        'image_height': config.image_height,
        'image_width': config.image_width,
    },
    'training_info': {
        'phase': 'pretrain',
        'dataset': 'synthetic_1M',
        'epochs_trained': config.pretrain_epochs,
        'best_val_cer': pretrain_trainer.best_val_cer,
    }
}, pretrain_full_path)
print(f"✅ Full checkpoint: {pretrain_full_path}")
print(f"   Size: {pretrain_full_path.stat().st_size / 1024 / 1024:.1f} MB")

# Save format 2: Weights only (lebih ringan, untuk inference)
pretrain_weights_path = MODELS_DIR / "pretrain_synthetic_weights.pt"
torch.save(model.state_dict(), pretrain_weights_path)
print(f"✅ Weights only: {pretrain_weights_path}")
print(f"   Size: {pretrain_weights_path.stat().st_size / 1024 / 1024:.1f} MB")

# Save format 3: ONNX (untuk deployment/inference cepat)
try:
    onnx_path = MODELS_DIR / "pretrain_synthetic.onnx"
    dummy_input = torch.randn(1, 1, config.image_height, config.image_width).to(device)
    model.eval()
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_path),
        input_names=['image'],
        output_names=['log_probs'],
        dynamic_axes={
            'image': {0: 'batch_size'},
            'log_probs': {1: 'batch_size'}
        },
        opset_version=11
    )
    print(f"✅ ONNX model: {onnx_path}")
    print(f"   Size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")
except Exception as e:
    print(f"⚠️ ONNX export failed: {e}")

print(f"\n📁 Models saved to: {MODELS_DIR}")
print(f"\n💡 Untuk eksperimen fine-tuning, gunakan:")
print(f"   model.load_state_dict(torch.load('{pretrain_weights_path}'))")


💾 SAVING PRE-TRAINED MODEL
✅ Full checkpoint: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\saved_models\pretrain_synthetic_full.pt
   Size: 38.4 MB
✅ Weights only: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\saved_models\pretrain_synthetic_weights.pt
   Size: 38.4 MB
⚠️ ONNX export failed: Unsupported: ONNX export of operator adaptive pooling, since output_size is not constant.. Please feel free to request support or submit a pull request on PyTorch GitHub: https://github.com/pytorch/pytorch/issues  [Caused by the value '113 defined in (%113 : Float(*, 512, 4, 128, strides=[262144, 512, 128, 1], requires_grad=1, device=cuda:0) = onnx::Relu(%input.68), scope: __main__.CRNN::/__main__.CNNBackbone::cnn/torch.nn.modules.container.Sequential::cnn/torch.nn.modules.activation.ReLU::cnn.24 # d:\d121221080_HTR_ta\Tugas_Akhir\htr_env\Lib\site-packages\torch\nn\functional.py:1702:0
)' (type 'Tensor') in the TorchScript graph. The containing node has kind 'onnx::Relu'.] 
    (node define

## Section 10: Phase 2 - Fine-tuning on IAM Data

In [20]:
# ============================================
# PHASE 2: FINE-TUNING ON IAM DATA
# ============================================

if not IAM_LOADED:
    print("⚠️ IAM dataset not loaded. Skipping Phase 2.")
else:
    print("\n" + "=" * 70)
    print("🚀 PHASE 2: FINE-TUNING ON IAM DATA")
    print("=" * 70)
    print(f"   Dataset: {len(iam_train_df):,} train, {len(iam_val_df):,} val")
    print(f"   Epochs: {config.finetune_epochs}")
    print(f"   Batch size: {config.finetune_batch_size}")
    print(f"   Learning rate: {config.finetune_lr}")
    print(f"   num_workers: 0 (Windows fix)")
    print("=" * 70)

    # Load best pre-trained model
    pretrain_checkpoint_path = CHECKPOINT_DIR / "pretrain_best.pt"
    if pretrain_checkpoint_path.exists():
        checkpoint = torch.load(pretrain_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"\n✅ Loaded pre-trained model from: pretrain_best.pt")
        print(f"   Pre-train best CER: {checkpoint['best_val_cer']*100:.2f}%")
    else:
        print(f"\n⚠️ Pre-trained model not found, using current weights")

    # Create IAM datasets
    iam_train_dataset = HTRDataset(
        iam_train_df,
        encoder,
        image_height=config.image_height,
        image_width=config.image_width,
        augment=config.augment_train,
        base_path=""
    )

    iam_val_dataset = HTRDataset(
        iam_val_df,
        encoder,
        image_height=config.image_height,
        image_width=config.image_width,
        augment=False,
        base_path=""
    )

    # Create dataloaders - num_workers=0 untuk Windows!
    iam_train_loader = DataLoader(
        iam_train_dataset,
        batch_size=config.finetune_batch_size,
        shuffle=True,
        num_workers=0,  # ⚠️ HARUS 0 untuk Windows!
        pin_memory=config.pin_memory,
        collate_fn=collate_fn,
        drop_last=True
    )

    iam_val_loader = DataLoader(
        iam_val_dataset,
        batch_size=config.finetune_batch_size,
        shuffle=False,
        num_workers=0,  # ⚠️ HARUS 0 untuk Windows!
        pin_memory=config.pin_memory,
        collate_fn=collate_fn
    )

    print(f"\n📦 DataLoaders created (num_workers=0):")
    print(f"   Train: {len(iam_train_loader):,} batches")
    print(f"   Val: {len(iam_val_loader):,} batches")

    # Create trainer for Phase 2
    finetune_trainer = HTRTrainer(
        model=model,
        encoder=encoder,
        config=config,
        device=device,
        checkpoint_dir=CHECKPOINT_DIR,
        log_dir=RESULTS_DIR
    )

    # Fine-tune!
    print("\n🔄 Starting fine-tuning...")
    finetune_history = finetune_trainer.fit(
        iam_train_loader,
        iam_val_loader,
        num_epochs=config.finetune_epochs,
        learning_rate=config.finetune_lr,
        checkpoint_prefix="finetune"
    )

    print("\n✅ Phase 2 (Fine-tuning) complete!")


🚀 PHASE 2: FINE-TUNING ON IAM DATA
   Dataset: 6,482 train, 976 val
   Epochs: 100
   Batch size: 32
   Learning rate: 0.0001
   num_workers: 0 (Windows fix)

✅ Loaded pre-trained model from: pretrain_best.pt
   Pre-train best CER: 1.78%

📦 DataLoaders created (num_workers=0):
   Train: 202 batches
   Val: 31 batches
✅ Trainer initialized
   Log dir: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\results
   Checkpoint dir: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\checkpoints

🔄 Starting fine-tuning...

🚀 STARTING TRAINING
   Epochs: 100
   Learning rate: 0.0001
   Device: cuda
   Checkpoints: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\checkpoints


Epoch 1/100 | LR: 1.00e-04



📊 Epoch 1 Summary:
   Train Loss: 2.3773
   Val Loss:   1.1062
   Val CER:    0.2593 (25.93%)
   Val WER:    0.6364 (63.64%)
   Time:       79.8s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'cIt aras a isplendid intepnetation Ef that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'rympathctic C C. hk nBamoman gane anothe'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'pant . rhe 7ait ef the cat soe noel chesen ,'

   🏆 New best model! CER: 0.2593
   💾 Saved: finetune_epoch_1.pt
   💾 Saved: finetune_best.pt

Epoch 2/100 | LR: 1.00e-04



📊 Epoch 2 Summary:
   Train Loss: 1.4539
   Val Loss:   0.8507
   Val CER:    0.2199 (21.99%)
   Val WER:    0.5784 (57.84%)
   Time:       51.4s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It as a isplendid intepnetation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'mympathctic C C. ihk nBamoman gane anothe'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'pert . rhe Pait of the cat soe soel choson ,'

   🏆 New best model! CER: 0.2199
   💾 Saved: finetune_epoch_2.pt
   💾 Saved: finetune_best.pt

Epoch 3/100 | LR: 1.00e-04



📊 Epoch 3 Summary:
   Train Loss: 1.2329
   Val Loss:   0.7364
   Val CER:    0.1978 (19.78%)
   Val WER:    0.5447 (54.47%)
   Time:       51.3s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a isplendid intepretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'mympathctic C C. Shk nEamoman gave anothe'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'pert . the auit of the cat soe wel choson ,'

   🏆 New best model! CER: 0.1978
   💾 Saved: finetune_epoch_3.pt
   💾 Saved: finetune_best.pt

Epoch 4/100 | LR: 1.00e-04



📊 Epoch 4 Summary:
   Train Loss: 1.1113
   Val Loss:   0.6651
   Val CER:    0.1804 (18.04%)
   Val WER:    0.5169 (51.69%)
   Time:       51.5s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a isplendid intespretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'mympaithctic C' C. Shk Eamoman gave anothe'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'pest . ithe auit of the cant soese wel choson ,'

   🏆 New best model! CER: 0.1804
   💾 Saved: finetune_epoch_4.pt
   💾 Saved: finetune_best.pt

Epoch 5/100 | LR: 1.00e-04


...


📊 Epoch 96 Summary:
   Train Loss: 0.2478
   Val Loss:   0.2542
   Val CER:    0.0771 (7.71%)
   Val WER:    0.2724 (27.24%)
   Time:       51.3s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a splendid interpretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'sympathetic ( 0 . Soul Ganeman gave another'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'part . ' The rust of the cast wore wel chosen .'
   💾 Saved: finetune_epoch_96.pt

Epoch 97/100 | LR: 1.00e-04



📊 Epoch 97 Summary:
   Train Loss: 0.2528
   Val Loss:   0.2544
   Val CER:    0.0764 (7.64%)
   Val WER:    0.2700 (27.00%)
   Time:       51.4s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a splendid interpretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'sympathetic ( 0 . Poul Ganeman gave another'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'part . ' The rust of the cast were wel chosen .'
   💾 Saved: finetune_epoch_97.pt

Epoch 98/100 | LR: 1.00e-04


   📉 Learning rate reduced: 1.00e-04 → 5.00e-05

📊 Epoch 98 Summary:
   Train Loss: 0.2511
   Val Loss:   0.2546
   Val CER:    0.0760 (7.60%)
   Val WER:    0.2698 (26.98%)
   Time:       51.3s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a splendid interpretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'sympathetic ( 0 . Soul Ganeman gave another'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'part . ' The rust of the cast wore wel chosen ,'

   🏆 New best model! CER: 0.0760
   💾 Saved: finetune_epoch_98.pt
   💾 Saved: finetune_best.pt

Epoch 99/100 | LR: 5.00e-05



📊 Epoch 99 Summary:
   Train Loss: 0.2438
   Val Loss:   0.2537
   Val CER:    0.0761 (7.61%)
   Val WER:    0.2694 (26.94%)
   Time:       51.3s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a splendid interpretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'sympathetic ( 0 . Soul Ganeman gave another'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'part . ' The rust of the cast wore wel chosen ,'
   💾 Saved: finetune_epoch_99.pt

Epoch 100/100 | LR: 5.00e-05



📊 Epoch 100 Summary:
   Train Loss: 0.2441
   Val Loss:   0.2541
   Val CER:    0.0759 (7.59%)
   Val WER:    0.2689 (26.89%)
   Time:       51.4s

📝 Sample Predictions:
   [1] GT:   'It was a splendid interpretation of the'
       Pred: 'It was a splendid interpretation of that'
   [2] GT:   'sympathetic C O . Paul Daneman gave another'
       Pred: 'sympathetic ( 0 . Soul Ganeman gave another'
   [3] GT:   'part . The rest of the cast were well chosen ,'
       Pred: 'part . ' The rust of the cast wore wel chosen .'

   🏆 New best model! CER: 0.0759
   💾 Saved: finetune_epoch_100.pt
   💾 Saved: finetune_best.pt
   💾 Saved: finetune_final.pt

✅ Training completed!
   Total time: 1:32:12.824977
   Best CER: 0.0759 at epoch 100

✅ Phase 2 (Fine-tuning) complete!


In [21]:
# ============================================
# 💾 SAVE FINE-TUNED MODEL (untuk eksperimen)
# ============================================

if IAM_LOADED:
    print("\n" + "=" * 60)
    print("💾 SAVING FINE-TUNED MODEL")
    print("=" * 60)
    
    # Load best fine-tuned model untuk disave
    best_finetune_path = CHECKPOINT_DIR / "finetune_best.pt"
    if best_finetune_path.exists():
        best_checkpoint = torch.load(best_finetune_path, map_location=device)
        model.load_state_dict(best_checkpoint['model_state_dict'])
        best_cer = best_checkpoint.get('best_val_cer', 0)
        best_epoch = best_checkpoint.get('best_epoch', 0)
    else:
        best_cer = finetune_trainer.best_val_cer
        best_epoch = finetune_trainer.best_epoch
    
    # Save format 1: Full checkpoint (untuk resume/analisis)
    finetune_full_path = MODELS_DIR / "finetune_iam_full.pt"
    torch.save({
        'model_state_dict': model.state_dict(),
        'encoder_charset': encoder.char_to_idx,
        'config': {
            'num_classes': encoder.num_classes,
            'cnn_output_channels': config.cnn_output_channels,
            'rnn_hidden_size': config.rnn_hidden_size,
            'rnn_num_layers': config.rnn_num_layers,
            'rnn_dropout': config.rnn_dropout,
            'bidirectional': config.bidirectional,
            'image_height': config.image_height,
            'image_width': config.image_width,
        },
        'training_info': {
            'phase': 'finetune',
            'pretrain_dataset': 'synthetic_1M',
            'finetune_dataset': 'IAM_lines',
            'epochs_trained': finetune_trainer.current_epoch + 1,
            'best_epoch': best_epoch,
            'best_val_cer': best_cer,
            'finetune_lr': config.finetune_lr,
            'finetune_batch_size': config.finetune_batch_size,
            'augment_train': config.augment_train,
        }
    }, finetune_full_path)
    print(f"✅ Full checkpoint: {finetune_full_path}")
    print(f"   Size: {finetune_full_path.stat().st_size / 1024 / 1024:.1f} MB")
    print(f"   Best CER: {best_cer*100:.2f}% (epoch {best_epoch})")
    
    # Save format 2: Weights only (untuk inference)
    finetune_weights_path = MODELS_DIR / "finetune_iam_weights.pt"
    torch.save(model.state_dict(), finetune_weights_path)
    print(f"✅ Weights only: {finetune_weights_path}")
    print(f"   Size: {finetune_weights_path.stat().st_size / 1024 / 1024:.1f} MB")
    
    # Save format 3: ONNX (untuk deployment)
    try:
        onnx_path = MODELS_DIR / "finetune_iam.onnx"
        dummy_input = torch.randn(1, 1, config.image_height, config.image_width).to(device)
        model.eval()
        torch.onnx.export(
            model,
            dummy_input,
            str(onnx_path),
            input_names=['image'],
            output_names=['log_probs'],
            dynamic_axes={
                'image': {0: 'batch_size'},
                'log_probs': {1: 'batch_size'}
            },
            opset_version=11
        )
        print(f"✅ ONNX model: {onnx_path}")
        print(f"   Size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")
    except Exception as e:
        print(f"⚠️ ONNX export failed: {e}")
    
    # Save IAM CSVs untuk eksperimen berikutnya
    iam_train_df.to_csv(OUTPUT_DIR / "iam_train.csv", index=False)
    iam_val_df.to_csv(OUTPUT_DIR / "iam_val.csv", index=False)
    iam_test_df.to_csv(OUTPUT_DIR / "iam_test.csv", index=False)
    print(f"\n✅ IAM CSVs saved for future experiments")
    
    print(f"\n📁 All models saved to: {MODELS_DIR}")
    
    # Summary
    print(f"\n" + "=" * 60)
    print("📋 SAVED FILES SUMMARY")
    print("=" * 60)
    print(f"\n🔷 Pre-trained Models (Synthetic):")
    print(f"   • pretrain_synthetic_full.pt   - Full checkpoint")
    print(f"   • pretrain_synthetic_weights.pt - Weights only")
    print(f"   • pretrain_synthetic.onnx      - ONNX format")
    print(f"\n🔷 Fine-tuned Models (IAM):")
    print(f"   • finetune_iam_full.pt   - Full checkpoint")
    print(f"   • finetune_iam_weights.pt - Weights only")
    print(f"   • finetune_iam.onnx      - ONNX format")
    print(f"\n🔷 Data Files:")
    print(f"   • iam_train.csv, iam_val.csv, iam_test.csv")
    print(f"   • charset.json")
    print(f"\n💡 Untuk eksperimen parameter berbeda:")
    print(f"   1. Load pretrain: torch.load('pretrain_synthetic_weights.pt')")
    print(f"   2. Ubah parameter (LR, batch size, augment)")
    print(f"   3. Jalankan fine-tuning ulang")
    print(f"   4. Bandingkan hasil CER/WER")


💾 SAVING FINE-TUNED MODEL
✅ Full checkpoint: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\saved_models\finetune_iam_full.pt
   Size: 38.4 MB
   Best CER: 7.59% (epoch 100)
✅ Weights only: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\saved_models\finetune_iam_weights.pt
   Size: 38.4 MB
⚠️ ONNX export failed: Unsupported: ONNX export of operator adaptive pooling, since output_size is not constant.. Please feel free to request support or submit a pull request on PyTorch GitHub: https://github.com/pytorch/pytorch/issues  [Caused by the value '113 defined in (%113 : Float(*, 512, 4, 128, strides=[262144, 512, 128, 1], requires_grad=1, device=cuda:0) = onnx::Relu(%input.68), scope: __main__.CRNN::/__main__.CNNBackbone::cnn/torch.nn.modules.container.Sequential::cnn/torch.nn.modules.activation.ReLU::cnn.24 # d:\d121221080_HTR_ta\Tugas_Akhir\htr_env\Lib\site-packages\torch\nn\functional.py:1702:0
)' (type 'Tensor') in the TorchScript graph. The containing node has kind 'onnx::Relu'.]

## Section 11: Evaluation on Test Set

In [22]:
# ============================================
# EVALUATE ON IAM TEST SET
# ============================================

if not IAM_LOADED:
    print("⚠️ IAM dataset not loaded. Skipping evaluation.")
else:
    print("\n" + "=" * 70)
    print("🔬 EVALUATION ON IAM TEST SET")
    print("=" * 70)
    print(f"   Test samples: {len(iam_test_df):,}")
    print("=" * 70)

    # Load best fine-tuned model
    finetune_checkpoint_path = CHECKPOINT_DIR / "finetune_best.pt"
    if finetune_checkpoint_path.exists():
        checkpoint = torch.load(finetune_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Loaded best fine-tuned model from epoch {checkpoint['best_epoch']}")
        print(f"   Best validation CER: {checkpoint['best_val_cer']*100:.2f}%")

    # Create test dataset and dataloader
    iam_test_dataset = HTRDataset(
        iam_test_df,
        encoder,
        image_height=config.image_height,
        image_width=config.image_width,
        augment=False,
        base_path=""
    )

    iam_test_loader = DataLoader(
        iam_test_dataset,
        batch_size=config.finetune_batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        collate_fn=collate_fn
    )

    # Evaluate
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for images, targets, target_lengths, texts in tqdm(iam_test_loader, desc="Evaluating"):
            images = images.to(device)
            
            log_probs = model(images)
            predictions = decode_predictions(log_probs, encoder)
            
            all_predictions.extend(predictions)
            all_targets.extend(texts)

    # Calculate metrics
    overall_cer = calculate_cer(all_predictions, all_targets)
    overall_wer = calculate_wer(all_predictions, all_targets)

    print(f"\n" + "=" * 60)
    print("📊 TEST RESULTS (IAM Test Set)")
    print("=" * 60)
    print(f"\n🎯 OVERALL METRICS:")
    print(f"   CER: {overall_cer:.4f} ({overall_cer*100:.2f}%)")
    print(f"   WER: {overall_wer:.4f} ({overall_wer*100:.2f}%)")

    # Save results
    test_results = {
        'overall_metrics': {
            'cer': overall_cer,
            'wer': overall_wer,
            'cer_percent': overall_cer * 100,
            'wer_percent': overall_wer * 100,
        },
        'dataset_info': {
            'total_samples': len(all_predictions),
        },
    }

    with open(RESULTS_DIR / "test_evaluation_results.json", 'w', encoding='utf-8') as f:
        json.dump(test_results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Results saved to: {RESULTS_DIR / 'test_evaluation_results.json'}")


🔬 EVALUATION ON IAM TEST SET
   Test samples: 2,915
✅ Loaded best fine-tuned model from epoch 100
   Best validation CER: 7.59%


Evaluating: 100%|██████████| 92/92 [00:23<00:00,  3.87it/s]


📊 TEST RESULTS (IAM Test Set)

🎯 OVERALL METRICS:
   CER: 0.0924 (9.24%)
   WER: 0.2884 (28.84%)

✅ Results saved to: D:\d121221080_HTR_ta\Tugas_Akhir\training_output\results\test_evaluation_results.json


## Summary

### 📁 Output Files:

#### Checkpoints (untuk resume training):
- `checkpoints/pretrain_best.pt` - Best model after pre-training
- `checkpoints/finetune_best.pt` - Best model after fine-tuning

#### Saved Models (untuk eksperimen):
- `saved_models/pretrain_synthetic_full.pt` - Pre-trained model (full)
- `saved_models/pretrain_synthetic_weights.pt` - Pre-trained model (weights only)
- `saved_models/pretrain_synthetic.onnx` - Pre-trained model (ONNX)
- `saved_models/finetune_iam_full.pt` - Fine-tuned model (full)
- `saved_models/finetune_iam_weights.pt` - Fine-tuned model (weights only)
- `saved_models/finetune_iam.onnx` - Fine-tuned model (ONNX)

#### Data & Config:
- `results/experiment_config.json` - Training configuration
- `results/test_evaluation_results.json` - Test results
- `results/training_history.json` - Training history
- `results/charset.json` - Character encoding
- `iam_train.csv`, `iam_val.csv`, `iam_test.csv` - IAM data splits

---

### 🔬 Cara Melakukan Eksperimen Parameter:

```python
# 1. Load pre-trained model
model = CRNN(num_classes=encoder.num_classes, ...)
model.load_state_dict(torch.load('saved_models/pretrain_synthetic_weights.pt'))

# 2. Ubah parameter fine-tuning
config.finetune_lr = 5e-5        # Coba LR berbeda
config.finetune_batch_size = 64  # Coba batch size berbeda
config.augment_train = False     # Coba tanpa augmentasi

# 3. Jalankan fine-tuning
# 4. Catat hasil CER/WER
# 5. Bandingkan dengan eksperimen lain
```

### Tips untuk Local Training:
1. **GPU Memory**: Kurangi batch size jika out of memory
2. **num_workers**: Set ke 0 jika ada masalah di Windows
3. **Resume Training**: Load checkpoint dan lanjutkan training
4. **Mixed Precision**: Tambahkan `torch.cuda.amp` untuk training lebih cepat